In [1]:
!pip install numpy matplotlib scipy torch torchvision

In [2]:
import os
import random
import gc
import numpy as np
from PIL import Image
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

In [3]:
import zipfile

zip_path = "dataset.zip"   # your uploaded file name
extract_path = "./T"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped successfully")

✅ Unzipped successfully


In [4]:
DATA_ROOT  = r"./Dataset_BUSI_with_GT"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4          # reduced for shared GPU
ACCUM_STEPS= 4          # effective batch = 16
NUM_EPOCHS = 50
LR         = 3e-4       # slightly lower — more stable with mixed precision

In [5]:
class BUSISegDataset(Dataset):
    def __init__(self, root, augment=False):
        self.imgs, self.masks = [], []
        self.augment = augment

        for folder in ["benign", "malignant", "normal"]:
            path = os.path.join(root, folder)
            for f in sorted(os.listdir(path)):
                if "_mask" not in f and f.endswith(".png"):
                    img_p  = os.path.join(path, f)
                    mask_p = os.path.join(path, f.replace(".png", "_mask.png"))
                    if os.path.exists(mask_p):
                        self.imgs.append(img_p)
                        self.masks.append(mask_p)

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    def __len__(self): return len(self.imgs)


In [6]:
def __getitem__(self, i):
        img  = Image.open(self.imgs[i]).convert("RGB")
        mask = Image.open(self.masks[i]).convert("L")

        img  = TF.resize(img,  IMAGE_SIZE)
        mask = TF.resize(mask, IMAGE_SIZE, interpolation=InterpolationMode.NEAREST)

        if self.augment:
            # Geometric — applied identically to img and mask
            if random.random() > 0.5:
                img, mask = TF.hflip(img), TF.hflip(mask)
            if random.random() > 0.5:
                img, mask = TF.vflip(img), TF.vflip(mask)

            angle = random.uniform(-25, 25)
            img   = TF.rotate(img,  angle)
            mask  = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            # Random crop & resize (simulates zoom)
            i_, j_, h_, w_ = transforms.RandomResizedCrop.get_params(
                img, scale=(0.75, 1.0), ratio=(0.9, 1.1)
            )
            img  = TF.resized_crop(img,  i_, j_, h_, w_, IMAGE_SIZE)
            mask = TF.resized_crop(mask, i_, j_, h_, w_, IMAGE_SIZE,
                                   interpolation=InterpolationMode.NEAREST)

            # Photometric — image only
            if random.random() > 0.3:
                img = TF.adjust_brightness(img, random.uniform(0.75, 1.25))
            if random.random() > 0.3:
                img = TF.adjust_contrast(img,   random.uniform(0.75, 1.25))
            if random.random() > 0.5:
                img = TF.adjust_gamma(img, random.uniform(0.8, 1.2))
            if random.random() > 0.5:
                img = TF.gaussian_blur(img, kernel_size=3)
        img  = TF.to_tensor(img)
        img  = self.normalize(img)
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()
        return img, mask     

In [7]:
# ---------------- METRICS ----------------
def binarize(x, thr=0.5): return (x > thr).float()

def dice_score(p, g):
    p = binarize(p)
    inter = (p * g).sum()
    return (2 * inter + 1e-7) / (p.sum() + g.sum() + 1e-7)

def iou_score(p, g):
    p = binarize(p)
    inter = (p * g).sum()
    union = (p + g - p * g).sum()
    return (inter + 1e-7) / (union + 1e-7)

def precision_score(p, g):
    p = binarize(p)
    TP = (p * g).sum()
    FP = (p * (1 - g)).sum()
    return (TP + 1e-7) / (TP + FP + 1e-7)
def sensitivity(p, g):
    p = binarize(p)
    TP = (p * g).sum()
    FN = ((1 - p) * g).sum()
    return (TP + 1e-7) / (TP + FN + 1e-7)

def specificity(p, g):
    p = binarize(p)
    TN = ((1 - p) * (1 - g)).sum()
    FP = (p * (1 - g)).sum()
    return (TN + 1e-7) / (TN + FP + 1e-7)

def pixel_accuracy(p, g):
    p = binarize(p)
    correct = (p == g).float().sum()
    return correct / g.numel()
def assd(p, g):
    p = binarize(p).squeeze().cpu().numpy()
    g = g.squeeze().cpu().numpy()
    if p.sum() == 0 or g.sum() == 0:
        return 0.0
    return ((distance_transform_edt(1 - p) * g).mean() +
            (distance_transform_edt(1 - g) * p).mean())    

In [8]:
# ---------------- LOSS ----------------
def dice_loss(p, g):
    p = p.view(p.size(0), -1)
    g = g.view(g.size(0), -1)
    inter = (p * g).sum(1)
    return 1 - ((2 * inter + 1e-7) / (p.sum(1) + g.sum(1) + 1e-7)).mean()

def focal_loss(p, g, alpha=0.8, gamma=2.0):
    bce = F.binary_cross_entropy(p, g, reduction='none')
    pt  = torch.exp(-bce)
    return (alpha * (1 - pt) ** gamma * bce).mean()

def tversky_loss(p, g, alpha=0.7, beta=0.3):
    """Penalises false negatives more — helps recall on small lesions."""
    p = p.view(p.size(0), -1)
    g = g.view(g.size(0), -1)
    TP = (p * g).sum(1)
    FP = (p * (1 - g)).sum(1)
    FN = ((1 - p) * g).sum(1)
    return 1 - ((TP + 1e-7) / (TP + alpha * FP + beta * FN + 1e-7)).mean()
def combo_loss(p, g):
    return focal_loss(p, g) + 0.7 * dice_loss(p, g) + 0.3 * tversky_loss(p, g)

In [9]:
# ---------------- ATTENTION GATE ----------------
class AttentionGate(nn.Module):
    """Soft attention on skip connections — focuses on lesion regions."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[-2:] != x1.shape[-2:]:
            g1 = F.interpolate(g1, size=x1.shape[-2:], mode='bilinear', align_corners=True)
        return x * self.psi(F.relu(g1 + x1, inplace=True))

In [10]:
# ---------------- CONV BLOCK ----------------
class Conv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.c = nn.Sequential(*layers)

    def forward(self, x): return self.c(x)


In [11]:
# ---------------- MODEL: Attention UNet++ ----------------
class AttUNetPP(nn.Module):
    def __init__(self, base=48):
        super().__init__()
        b = base
        # Encoder
        self.e1 = Conv(3,     b)
        self.e2 = Conv(b,   2*b)
        self.e3 = Conv(2*b, 4*b)
        self.e4 = Conv(4*b, 8*b)
        self.e5 = Conv(8*b,16*b, dropout=0.15)

        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Attention gates on each encoder skip
        self.ag1 = AttentionGate(2*b,  b,   b//2)
        self.ag2 = AttentionGate(4*b, 2*b,  b)
        self.ag3 = AttentionGate(8*b, 4*b,  2*b)
        self.ag4 = AttentionGate(16*b,8*b,  4*b)
        # Dense decoder nodes (UNet++ notation: x_{i,j})
        self.x01 = Conv(b   + 2*b,       b)
        self.x02 = Conv(b*2 + 2*b,       b)
        self.x03 = Conv(b*3 + 2*b,       b)
        self.x04 = Conv(b*4 + 2*b,       b)

        self.x11 = Conv(2*b + 4*b,       2*b)
        self.x12 = Conv(2*b*2 + 4*b,     2*b)
        self.x13 = Conv(2*b*3 + 4*b,     2*b)

        self.x21 = Conv(4*b + 8*b,       4*b)
        self.x22 = Conv(4*b*2 + 8*b,     4*b)

        self.x31 = Conv(8*b + 16*b,      8*b)

        # Deep supervision output heads
        self.out1 = nn.Conv2d(b, 1, 1)
        self.out2 = nn.Conv2d(b, 1, 1)
        self.out3 = nn.Conv2d(b, 1, 1)
        self.out4 = nn.Conv2d(b, 1, 1)
    def forward(self, x):
        # Encoder
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        e5 = self.e5(self.pool(e4))

        # Dense decoder with attention-gated skip connections
        x31 = self.x31(torch.cat([self.ag4(self.up(e5), e4), self.up(e5)], 1))

        x21 = self.x21(torch.cat([self.ag3(self.up(e4), e3), self.up(e4)], 1))
        x22 = self.x22(torch.cat([self.ag3(self.up(x31), e3), x21, self.up(x31)], 1))

        x11 = self.x11(torch.cat([self.ag2(self.up(e3), e2), self.up(e3)], 1))
        x12 = self.x12(torch.cat([self.ag2(self.up(x21), e2), x11, self.up(x21)], 1))
        x13 = self.x13(torch.cat([self.ag2(self.up(x22), e2), x11, x12, self.up(x22)], 1))

        x01 = self.x01(torch.cat([self.ag1(self.up(e2), e1), self.up(e2)], 1))
        x02 = self.x02(torch.cat([self.ag1(self.up(x11), e1), x01, self.up(x11)], 1))
        x03 = self.x03(torch.cat([self.ag1(self.up(x12), e1), x01, x02, self.up(x12)], 1))
        x04 = self.x04(torch.cat([self.ag1(self.up(x13), e1), x01, x02, x03, self.up(x13)], 1))
        o1 = torch.sigmoid(self.out1(x01))
        o2 = torch.sigmoid(self.out2(x02))
        o3 = torch.sigmoid(self.out3(x03))
        o4 = torch.sigmoid(self.out4(x04))

        if self.training:
            return o1, o2, o3, o4
        return o4

In [12]:
# ---------------- TRAIN ----------------
def evaluate_batch(p, y):
    """Return dict of metrics for one batch (tensors on any device)."""
    return {
        "dice":  dice_score(p, y).item(),
        "iou":   iou_score(p, y).item(),
        "prec":  precision_score(p, y).item(),
        "acc":   pixel_accuracy(p, y).item(),
    }

import sys

def train(model, train_loader, val_loader):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR * 10,
        steps_per_epoch=max(1, len(train_loader) // ACCUM_STEPS),
        epochs=NUM_EPOCHS,
        pct_start=0.15,
        anneal_strategy='cos',
        div_factor=10,
        final_div_factor=100,
    )
    scaler   = torch.amp.GradScaler('cuda')
    best_val = float('inf')
    patience, wait = 12, 0

    for epoch in range(NUM_EPOCHS):
        # ---------- TRAIN ----------
        model.train()
        loss_sum = 0.0
        tr_metrics = {"dice": 0, "iou": 0, "prec": 0, "acc": 0}
        opt.zero_grad()

        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                o1, o2, o3, o4 = model(x)
                loss = (0.50 * combo_loss(o4, y) +
                        0.25 * combo_loss(o3, y) +
                        0.15 * combo_loss(o2, y) +
                        0.10 * combo_loss(o1, y))
                loss = loss / ACCUM_STEPS

            scaler.scale(loss).backward()

            with torch.no_grad():
                m = evaluate_batch(o4.float(), y)
                for k in tr_metrics:
                    tr_metrics[k] += m[k]

            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                scheduler.step()

            loss_sum += loss.item() * ACCUM_STEPS
            del x, y, o1, o2, o3, o4, loss

        n_tr = len(train_loader)
        for k in tr_metrics:
            tr_metrics[k] /= n_tr

        # ---------- VALIDATE ----------
        model.eval()
        val_loss = 0.0
        val_metrics = {"dice": 0, "iou": 0, "prec": 0, "acc": 0}

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    p = model(x)
                    val_loss += combo_loss(p.float(), y).item()
                m = evaluate_batch(p.float(), y)
                for k in val_metrics:
                    val_metrics[k] += m[k]
                del x, y, p

        n_val = len(val_loader)
        for k in val_metrics:
            val_metrics[k] /= n_val

        torch.cuda.empty_cache()

        # ---- Print with flush=True so Jupyter shows it immediately ----
        print(
            f"Ep {epoch+1:03d}/{NUM_EPOCHS} | "
            f"Loss {loss_sum/n_tr:.4f} | "
            f"Tr  Acc {tr_metrics['acc']*100:.1f}%  "
            f"Prec {tr_metrics['prec']*100:.1f}%  "
            f"Dice {tr_metrics['dice']*100:.1f}%  "
            f"IoU {tr_metrics['iou']*100:.1f}% | "
            f"Val Acc {val_metrics['acc']*100:.1f}%  "
            f"Prec {val_metrics['prec']*100:.1f}%  "
            f"Dice {val_metrics['dice']*100:.1f}%  "
            f"IoU {val_metrics['iou']*100:.1f}%",
            flush=True          # <-- key fix
        )
        sys.stdout.flush()      # belt-and-suspenders for Jupyter

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "best_model.pth")
            wait = 0
            print("  ✓ saved best model", flush=True)
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}.", flush=True)
                break

    model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
    print("Best model restored.", flush=True)

In [13]:
# ---------------- MAIN ----------------
if __name__ == "__main__":
    torch.cuda.empty_cache(); gc.collect()

    # Build one master list of (img_path, mask_path) pairs
    all_imgs, all_masks = [], []
    for folder in ["benign", "malignant", "normal"]:
        path = os.path.join(DATA_ROOT, folder)
        for f in sorted(os.listdir(path)):
            if "_mask" not in f and f.endswith(".png"):
                img_p  = os.path.join(path, f)
                mask_p = os.path.join(path, f.replace(".png", "_mask.png"))
                if os.path.exists(mask_p):
                    all_imgs.append(img_p)
                    all_masks.append(mask_p)

    N = len(all_imgs)
    idx = list(range(N))
    random.shuffle(idx)

    train_n = int(0.70 * N)
    val_n   = int(0.15 * N)
    test_n  = N - train_n - val_n

    train_idx = idx[:train_n]
    val_idx   = idx[train_n : train_n + val_n]
    test_idx  = idx[train_n + val_n:]

    # Pass pre-split path lists directly — no Subset wrapping needed
    class SplitDataset(Dataset):
        def __init__(self, imgs, masks, augment=False):
            self.imgs    = imgs
            self.masks   = masks
            self.augment = augment
            self.normalize = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            )

        def __len__(self): return len(self.imgs)

        def __getitem__(self, i):
            img  = Image.open(self.imgs[i]).convert("RGB")
            mask = Image.open(self.masks[i]).convert("L")

            img  = TF.resize(img,  IMAGE_SIZE)
            mask = TF.resize(mask, IMAGE_SIZE,
                             interpolation=InterpolationMode.NEAREST)

            if self.augment:
                if random.random() > 0.5:
                    img, mask = TF.hflip(img), TF.hflip(mask)
                if random.random() > 0.5:
                    img, mask = TF.vflip(img), TF.vflip(mask)

                angle = random.uniform(-25, 25)
                img   = TF.rotate(img,  angle)
                mask  = TF.rotate(mask, angle,
                                  interpolation=InterpolationMode.NEAREST)

                i_, j_, h_, w_ = transforms.RandomResizedCrop.get_params(
                    img, scale=(0.75, 1.0), ratio=(0.9, 1.1)
                )
                img  = TF.resized_crop(img,  i_, j_, h_, w_, IMAGE_SIZE)
                mask = TF.resized_crop(mask, i_, j_, h_, w_, IMAGE_SIZE,
                                       interpolation=InterpolationMode.NEAREST)

                if random.random() > 0.3:
                    img = TF.adjust_brightness(img, random.uniform(0.75, 1.25))
                if random.random() > 0.3:
                    img = TF.adjust_contrast(img, random.uniform(0.75, 1.25))
                if random.random() > 0.5:
                    img = TF.adjust_gamma(img, random.uniform(0.8, 1.2))
                if random.random() > 0.5:
                    img = TF.gaussian_blur(img, kernel_size=3)

            img  = TF.to_tensor(img)
            img  = self.normalize(img)
            mask = TF.to_tensor(mask)
            mask = (mask > 0.5).float()
            return img, mask

    train_ds = SplitDataset([all_imgs[i] for i in train_idx],
                             [all_masks[i] for i in train_idx], augment=True)
    val_ds   = SplitDataset([all_imgs[i] for i in val_idx],
                             [all_masks[i] for i in val_idx],   augment=False)
    test_ds  = SplitDataset([all_imgs[i] for i in test_idx],
                             [all_masks[i] for i in test_idx],  augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                              num_workers=0)

    model = AttUNetPP(base=48).to(DEVICE)
    print(f"Device         : {DEVICE}")
    print(f"Parameters     : {sum(p.numel() for p in model.parameters()):,}")
    print(f"Train/Val/Test : {train_n}/{val_n}/{test_n}\n")

    train(model, train_loader, val_loader)

    # ----------- FINAL TEST METRICS -----------
    dice_l, iou_l, prec_l, sens_l, spec_l, acc_l, assd_l = [], [], [], [], [], [], []

    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                p = model(x)
            p = p.float()

            dice_l.append(dice_score(p, y).item())
            iou_l.append(iou_score(p, y).item())
            prec_l.append(precision_score(p, y).item())
            sens_l.append(sensitivity(p, y).item())
            spec_l.append(specificity(p, y).item())
            acc_l.append(pixel_accuracy(p, y).item())
            assd_l.append(assd(p, y))
            del x, y, p

    print("\n========== FINAL TEST RESULTS ==========")
    print(f"Pixel Accuracy : {np.mean(acc_l)*100:.2f}%")
    print(f"Precision      : {np.mean(prec_l)*100:.2f}%")
    print(f"Dice (F1)      : {np.mean(dice_l)*100:.2f}%")
    print(f"Jaccard (IoU)  : {np.mean(iou_l)*100:.2f}%")
    print(f"Sensitivity    : {np.mean(sens_l)*100:.2f}%")
    print(f"Specificity    : {np.mean(spec_l)*100:.2f}%")
    print(f"ASSD           : {np.mean(assd_l):.4f}")
    print("========================================")

Device         : cuda
Parameters     : 20,898,852
Train/Val/Test : 546/117/117



RuntimeError: torch.nn.functional.binary_cross_entropy and torch.nn.BCELoss are unsafe to autocast.
Many models use a sigmoid layer right before the binary cross entropy layer.
In this case, combine the two layers using torch.nn.functional.binary_cross_entropy_with_logits
or torch.nn.BCEWithLogitsLoss.  binary_cross_entropy_with_logits and BCEWithLogits are
safe to autocast.